## Transformación y limpieza de datos

### Objetivo
Este notebook aplica las transformaciones necesarias sobre los datos extraídos para prepararlos para su carga en la base de datos analítica.

Las operaciones incluyen:
- **Tipado de columnas**: conversión de IDs numéricos a cadena (`str`) para tratarlos como variables categóricas.
- **Mapeo de territorios**: asignación del `id_territorio` desde la tabla dimensión a los DataFrames de hechos (constituidas y disueltas), y normalización de nombres (minúsculas, guiones bajos).
- **Normalización de texto**: estandarización de nombres de sectores, meses, razones de disolución y tipos de medida.
- **Eliminación de duplicados**: filtrado de filas "Mercantiles" que agregan información ya presente en los desgloses por tipo societario.
- **Abreviaturas**: conversión de tipos societarios a siglas (S.A., S.L., S.Com./S.C.).

### Metodología
1. **Carga** de los CSV desde `../files/data_raw/`.
2. **Transformaciones** aplicadas mediante funciones del módulo `src.transformation` y mapeos manuales.
3. **Exportación** de los datasets procesados a `../files/data_processed/` para su consumo en la fase de carga.

In [1]:
# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

import pandas as pd
# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación del módulo de transformacion
from src.transformation import trans_str
from src.transformation import trans_normal
# Configuración para visualizar todas las columnas
pd.set_option('display.max_columns', None)

Abrimos los ficheros

In [2]:
df_empr_const = pd.read_csv('../files/data_raw/empresas_constituidas.csv')
df_empr_dis = pd.read_csv('../files/data_raw/empresas_disueltas.csv')
df_ipc = pd.read_csv('../files/data_raw/ipc.csv')
df_sectores_ipc = pd.read_csv('../files/data_raw/sectores_ipc.csv')
df_territorio = pd.read_csv('../files/data_raw/territorio.csv')
df_tiempo = pd.read_csv('../files/data_raw/tiempo.csv')
df_tipo_medida = pd.read_csv('../files/data_raw/tipo_medida.csv')

Transformamos id's, año y mes a string ya que son falsas numericas (actuan como categoricas)

In [3]:
# Listas de columnas a transformar de cada DataFrame
lista_const = ['id_const', 'id_tiempo']
lista_dis = ['id_dis', 'id_tiempo']
lista_ipc = ['id_tiempo', 'id_territorio', 'id_sector', 'id_medida']
lista_sector_ipc = ['id_sector']
lista_territorio = ['id_territorio']
lista_tiempo = ['id_tiempo', 'anio', 'mes']
lista_tipo_medida = ['id_medida']

In [4]:
trans_str.int_a_str(df_empr_const, lista_const)
trans_str.int_a_str(df_empr_dis, lista_dis)
trans_str.int_a_str(df_ipc, lista_ipc)
trans_str.int_a_str(df_sectores_ipc, lista_sector_ipc)
trans_str.int_a_str(df_territorio, lista_territorio)
trans_str.int_a_str(df_tiempo, lista_tiempo)
trans_str.int_a_str(df_tipo_medida, lista_tipo_medida)

,id_medida,nombre_medida
0,1,Índice
1,2,Variación mensual
2,3,Variación anual
3,4,Variación en lo que va de año


In [5]:
df_empr_const.info()

<class 'pandas.DataFrame'>
RangeIndex: 16720 entries, 0 to 16719
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_const           16720 non-null  str  
 1   territorio         16720 non-null  str  
 2   id_tiempo          16720 non-null  str  
 3   tipo               16720 non-null  str  
 4   numero_sociedades  16720 non-null  int64
 5   capital            16720 non-null  int64
dtypes: int64(2), str(4)
memory usage: 783.9 KB


In [6]:
df_empr_dis.info()

<class 'pandas.DataFrame'>
RangeIndex: 12540 entries, 0 to 12539
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_dis             12540 non-null  str  
 1   territorio         12540 non-null  str  
 2   id_tiempo          12540 non-null  str  
 3   razon              12540 non-null  str  
 4   numero_sociedades  12540 non-null  int64
dtypes: int64(1), str(4)
memory usage: 490.0 KB


In [7]:
df_ipc.info()

<class 'pandas.DataFrame'>
RangeIndex: 328162 entries, 0 to 328161
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   id_tiempo      328162 non-null  str    
 1   id_territorio  328162 non-null  str    
 2   id_sector      328162 non-null  str    
 3   id_medida      328162 non-null  str    
 4   valor_ipc      328162 non-null  float64
dtypes: float64(1), str(4)
memory usage: 12.5 MB


In [8]:
df_sectores_ipc.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_sector      14 non-null     str  
 1   nombre_sector  14 non-null     str  
dtypes: str(2)
memory usage: 356.0 bytes


In [9]:
df_territorio.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_territorio      20 non-null     str  
 1   nombre_territorio  20 non-null     str  
dtypes: str(2)
memory usage: 452.0 bytes


In [10]:
df_territorio.head()

,id_territorio,nombre_territorio
0,1,Nacional
1,2,Andalucía
2,3,Aragón
3,4,"Asturias, Principado de"
4,5,"Balears, Illes"


In [11]:
df_tiempo.info()

<class 'pandas.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id_tiempo   294 non-null    str  
 1   anio        294 non-null    str  
 2   mes         294 non-null    str  
 3   nombre_mes  294 non-null    str  
dtypes: str(4)
memory usage: 9.3 KB


In [12]:
df_tipo_medida.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_medida      4 non-null      str  
 1   nombre_medida  4 non-null      str  
dtypes: str(2)
memory usage: 196.0 bytes


- Cambiar los Nombres de las comunidades autónomas, sin acentos, con minúsculas y separación con guión bajo ('_')
- DataFrames: empresas_constituidas, empresas_disueltas y territorio

In [13]:
# Creamos el diccionario de mapeo con tus especificaciones exactas
mapeo_territorios_nombre = {
    "Nacional": "nacional",
    "Andalucía": "andalucia",
    "Aragón": "aragon",
    "Asturias, Principado de": "principado_de_asturias",
    "Balears, Illes": "islas_baleares",
    "Canarias": "canarias",
    "Cantabria": "cantabria",
    "Castilla y León": "castilla_y_leon",
    "Castilla - La Mancha": "castilla_la_mancha",
    "Cataluña": "cataluna",
    "Comunitat Valenciana": "comunidad_valenciana",
    "Extremadura": "extremadura",
    "Galicia": "galicia",
    "Madrid, Comunidad de": "comunidad_de_madrid",
    "Murcia, Región de": "region_de_murcia",
    "Navarra, Comunidad Foral de": "comunidad_foral_de_navarra",
    "País Vasco": "pais_vasco",
    "Rioja, La": "la_rioja",
    "Ceuta": "ceuta",
    "Melilla": "melilla"
}

In [14]:
# Diccionario territorio -> id_territorio, usando el propio df_territorio
mapeo_territorios = dict(zip(df_territorio['nombre_territorio'], df_territorio['id_territorio']))

# Aplicar el mapeo
df_empr_const['id_territorio'] = df_empr_const['territorio'].map(mapeo_territorios)
df_empr_dis['id_territorio'] = df_empr_dis['territorio'].map(mapeo_territorios)

In [15]:
df_empr_const.sample(10)

,id_const,territorio,id_tiempo,tipo,numero_sociedades,capital,id_territorio
15400,15401,"Murcia, Región de",202604,S. Comanditarias y S. Colectivas,0,0,15
2408,2409,Extremadura,200812,Mercantiles,81,1718000,12
7275,7276,"Navarra, Comunidad Foral de",202501,Sociedades anónimas,1,1200000,16
2357,2358,Extremadura,201303,Mercantiles,102,1831000,12
3503,3504,País Vasco,200905,Mercantiles,276,12188000,17
4604,4605,Aragón,200904,Sociedades anónimas,5,457000,3
4638,4639,"Asturias, Principado de",202410,Sociedades anónimas,0,0,4
12204,12205,Ceuta,201708,Sociedades de responsabilidad limitada,8,33000,19
5384,5385,Cantabria,201708,Sociedades anónimas,0,0,7
10580,10581,Extremadura,202408,Sociedades de responsabilidad limitada,89,2955000,12


In [16]:
df_empr_dis.sample(10)

,id_dis,territorio,id_tiempo,razon,numero_sociedades,id_territorio
7938,7939,Ceuta,202410,Por fusión,0,19
7028,7029,"Madrid, Comunidad de",200812,Por fusión,71,14
11903,11904,"Rioja, La",202405,Otras,0,18
8462,8463,Andalucía,201710,Otras,38,2
760,761,"Balears, Illes",201712,Voluntaria,102,5
6926,6927,"Madrid, Comunidad de",201706,Por fusión,30,14
3655,3656,"Rioja, La",201501,Voluntaria,22,18
8849,8850,"Asturias, Principado de",202203,Otras,6,4
838,839,"Balears, Illes",201106,Voluntaria,37,5
8744,8745,Aragón,201208,Otras,11,3


In [17]:
df_empr_dis.drop(columns=["territorio"],inplace=True)

In [18]:
df_empr_const.drop(columns=["territorio"],inplace=True)

In [19]:
#df_empr_const['territorio'] = df_empr_const['territorio'].replace(mapeo_territorios)
#df_empr_dis['territorio'] = df_empr_dis['territorio'].replace(mapeo_territorios)
df_territorio['nombre_territorio'] = df_territorio['nombre_territorio'].replace(mapeo_territorios_nombre)

In [20]:
df_territorio['nombre_territorio'].unique()

<StringArray>
[                  'nacional',                  'andalucia',
                     'aragon',     'principado_de_asturias',
             'islas_baleares',                   'canarias',
                  'cantabria',            'castilla_y_leon',
         'castilla_la_mancha',                   'cataluna',
       'comunidad_valenciana',                'extremadura',
                    'galicia',        'comunidad_de_madrid',
           'region_de_murcia', 'comunidad_foral_de_navarra',
                 'pais_vasco',                   'la_rioja',
                      'ceuta',                    'melilla']
Length: 20, dtype: str

Normalizamos resto de columnas (minúsculas, separación con guión bajo ('_'))

In [21]:
# Así se aplica una función a los DATOS de una columna
df_empr_dis["razon"] = df_empr_dis["razon"].apply(trans_normal.normalizar_col)
df_sectores_ipc["nombre_sector"] = df_sectores_ipc["nombre_sector"].apply(trans_normal.normalizar_col)
df_tiempo["nombre_mes"] = df_tiempo["nombre_mes"].apply(trans_normal.normalizar_col)
df_tipo_medida["nombre_medida"] = df_tipo_medida["nombre_medida"].apply(trans_normal.normalizar_col)

In [22]:
df_empr_dis.sample(10)

,id_dis,id_tiempo,razon,numero_sociedades,id_territorio
9150,9151,201506,otras,3,5
11336,11337,201608,otras,2,15
347,348,201509,voluntaria,28,3
10829,10830,202203,otras,16,13
9365,9366,201511,otras,9,6
10399,10400,202105,otras,47,11
12038,12039,201302,otras,2,18
2834,2835,201002,voluntaria,303,14
5668,5669,201204,por_fusion,1,8
4161,4162,200907,voluntaria,0,20


In [23]:
df_sectores_ipc.sample(10)

,id_sector,nombre_sector
13,14,cuidado_personal_proteccion_social_y_bienes_y_...
0,1,indice_general
3,4,vestido_y_calzado
11,12,restaurantes_y_servicios_de_alojamiento
8,9,informacion_y_comunicaciones
2,3,bebidas_alcoholicas_y_tabaco
5,6,muebles_articulos_del_hogar_y_articulos_para_e...
10,11,ensenanza
12,13,seguros_y_servicios_financieros
4,5,vivienda_agua_electricidad_gas_y_otros_combust...


In [24]:
df_tiempo.sample(10)

,id_tiempo,anio,mes,nombre_mes
59,202106,2021,6,junio
261,200408,2004,8,agosto
121,201604,2016,4,abril
105,201708,2017,8,agosto
15,202502,2025,2,febrero
82,201907,2019,7,julio
192,201005,2010,5,mayo
23,202406,2024,6,junio
237,200608,2006,8,agosto
234,200611,2006,11,noviembre


In [25]:
df_tipo_medida.sample(4)

,id_medida,nombre_medida
1,2,variacion_mensual
2,3,variacion_anual
3,4,variacion_en_lo_que_va_de_ano
0,1,indice


Eliminar en empresas_contituidas las tipo mercantiles, son sumatorias del resto de tipo y nos duplican los datos

In [26]:
# Eliminamos las filas que contienen "Mercantiles"
df_empr_const = df_empr_const[df_empr_const['tipo'] != 'Mercantiles']

In [27]:
df_empr_const['tipo'].unique()

<StringArray>
[                   'Sociedades anónimas',
 'Sociedades de responsabilidad limitada',
       'S. Comanditarias y S. Colectivas']
Length: 3, dtype: str

In [28]:
df_empr_const.shape

(12540, 6)

Cambiamos el nombre de las sociedades por sus acrónimos

In [29]:
# Creamos el diccionario de mapeo con tus especificaciones exactas
dicc_siglas = {
    'Sociedades de responsabilidad limitada': 'S.L.',
    'Sociedades anónimas': 'S.A.',
    'S. Comanditarias y S. Colectivas': 'S.Com./S.C.'
}

In [30]:
# Aplicamos el cambio a la columna 'tipo'
df_empr_const['tipo'] = df_empr_const['tipo'].replace(dicc_siglas)

In [31]:
df_empr_const['tipo'].unique()

<StringArray>
['S.A.', 'S.L.', 'S.Com./S.C.']
Length: 3, dtype: str

Guardamos los csv's procesados

In [32]:
df_empr_const.to_csv('../files/data_processed/empresas_constituidas.csv', index=False)
df_empr_dis.to_csv('../files/data_processed/empresas_disueltas.csv', index=False)
df_ipc.to_csv('../files/data_processed/ipc.csv', index=False)
df_sectores_ipc.to_csv('../files/data_processed/sectores_ipc.csv', index=False)
df_territorio.to_csv('../files/data_processed/territorio.csv', index=False)
df_tiempo.to_csv('../files/data_processed/tiempo.csv', index=False)
df_tipo_medida.to_csv('../files/data_processed/tipo_medida.csv', index=False)

In [1]:
# NOTA: Los nombres normalizados están pensados para uso en BD y código.
# Para presentación en PowerBI, se recomienda crear columnas display 
# con los nombres originales del INE usando Data Category o DAX.